# Leyendo base de datos

In [ ]:
import pandas as pd

dataset_entrenamiento = pd.read_csv('../data/train_clean.csv')
dataset_prueba = pd.read_csv('../data/test_clean.csv')
dataset_OOT = pd.read_csv('../data/oot_clean.csv')

# Creando X & Y

In [ ]:
X_entrenamiento = dataset_entrenamiento.drop(columns=['es_fraude'])
y_entrenamiento = dataset_entrenamiento['es_fraude']

X_prueba = dataset_prueba.drop(columns=['es_fraude'])
y_prueba = dataset_prueba['es_fraude']

X_oot = dataset_OOT.drop(columns=['es_fraude'])
y_oot = dataset_OOT['es_fraude']

print("Dimensiones:")
print(f"Entrenamiento -> X: {X_entrenamiento.shape}, y: {y_entrenamiento.shape}")
print(f"Prueba        -> X: {X_prueba.shape}, y: {y_prueba.shape}")
print(f"OOT           -> X: {X_oot.shape}, y: {y_oot.shape}")

# K-1 Dummies

In [ ]:
cols_cat = [col for col in X_entrenamiento.columns if col.startswith('cat_')]
cols_job = [col for col in X_entrenamiento.columns if col.startswith('job_')]
cols_estado = [col for col in X_entrenamiento.columns if col.startswith('estado_')]

ref_cat = X_entrenamiento[cols_cat].sum().idxmax()
ref_job = X_entrenamiento[cols_job].sum().idxmax()
ref_estado = X_entrenamiento[cols_estado].sum().idxmax()

print("Categorías base por repetición para cada variable categórica en general:")
print(f"- Categoría de transacción: {ref_cat}")
print(f"- Categoría de empleo: {ref_job}")
print(f"- Estado: {ref_estado}")

In [ ]:
X_entrenamiento = X_entrenamiento.drop(columns=[ref_cat, ref_job, ref_estado])
X_prueba = X_prueba.drop(columns=[ref_cat, ref_job, ref_estado])
X_oot = X_oot.drop(columns=[ref_cat, ref_job, ref_estado])

In [ ]:
X_entrenamiento.head()

In [ ]:
X_entrenamiento.info()

# Función objetivo

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import optuna

In [ ]:
def objective(intento):
    
    # regularización lambda
    c_param = intento.suggest_float('C', 0.001, 10.0, log=True)
    
    # peso sugerido
    class_weight_param = intento.suggest_categorical('class_weight', [None, 'balanced'])
    
    # modelo
    modelo = LogisticRegression(
        C=c_param,
        class_weight=class_weight_param,
        max_iter=1000, # Alto para asegurar que 1.2M de registros converjan
        random_state=42
    )
    
    # entrenamiento
    modelo.fit(X_entrenamiento, y_entrenamiento)
    
    # predicciones
    y_pred_proba = modelo.predict_proba(X_prueba)[:, 1]
    
    # AUC
    auc = roc_auc_score(y_prueba, y_pred_proba)
    return auc

In [ ]:
import optuna

# maximización del AUC
estudio = optuna.create_study(direction='maximize')

print("Iniciando la búsqueda de hiperparámetros")

estudio.optimize(objective, n_trials=10)

print("\nBúsqueda lista")
print(f"Mejor AUC encontrado en Prueba: {estudio.best_value:.4f}")
print(f"Mejores hiperparámetros: {estudio.best_params}")

In [ ]:
modelo_logit_optimo = LogisticRegression(
    **estudio.best_params,
    max_iter=1000,
    random_state=42
)

print("Entrenando el modelo logit óptimo")
modelo_logit_optimo.fit(X_entrenamiento, y_entrenamiento)

# Mejor umbral f2_score

In [ ]:
import numpy as np
from sklearn.metrics import fbeta_score

mejor_umbral = 0.5
mejor_f2 = 0.0

print("Buscando el mejor umbral enfocado en Recall")

for umbral in np.arange(0.01, 1.0, 0.01):
    
    prediccion_temporal = (y_proba_prueba >= umbral).astype(int)
    
    f2_temporal = fbeta_score(y_prueba, prediccion_temporal, beta=2, zero_division=0)
    
    if f2_temporal > mejor_f2:
        mejor_f2 = f2_temporal
        mejor_umbral = umbral 

print(f" Umbral ganador: {mejor_umbral:.2f}")
print(f" F2-Score máximo alcanzado: {mejor_f2:.5f}")

# Desempeño del modelo

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, fbeta_score

print("="*50)
print("   RESULTADOS PRUEBA")
print("="*50)

y_pred_prueba_optimo = (y_proba_prueba >= mejor_umbral).astype(int)

print("\nMatriz de Confusión logit en Prueba:")
print(confusion_matrix(y_prueba, y_pred_prueba_optimo))

print("\nDesempeño modelo logit en Prueba:")
print(classification_report(y_prueba, y_pred_prueba_optimo, digits=5))

print("\n" + "="*50)
print("   RESULTADOS OOT")
print("="*50)

y_proba_oot = modelo_logit_optimo.predict_proba(X_oot)[:, 1]
y_pred_oot_optimo = (y_proba_oot >= mejor_umbral).astype(int)

print("\nMatriz de Confusión logit en OOT:")
print(confusion_matrix(y_oot, y_pred_oot_optimo))

print("\nDesempeño modelo logit en OOT:")
print(classification_report(y_oot, y_pred_oot_optimo, digits=5))